# **Импорт библиотек**

In [ ]:
import numpy as np
import os
import re
import time
import matplotlib.pyplot as plt
import gdown
from IPython.display import display

# Инструменты Keras
from tensorflow.keras import utils
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, Dropout, SpatialDropout1D, BatchNormalization,
                                     Embedding, Flatten, Activation, SimpleRNN, GRU,
                                     LSTM, Bidirectional, Conv1D, MaxPooling1D, GlobalMaxPooling1D)
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

%matplotlib inline

# Контекстный менеджер для замера времени
class timex:
    def __enter__(self):
        self.t = time.time()
        return self
    def __exit__(self, type, value, traceback):
        print('Время обработки: {:.2f} с'.format(time.time() - self.t))

# **Загрузка и подготовка набора данных**

In [ ]:
# Загрузка архива
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip', None, quiet=True)
!unzip -qo writers.zip -d writers/

# Константы для загрузки
FILE_DIR  = 'writers'
SIG_TRAIN = 'обучающая'
SIG_TEST  = 'тестовая'

CLASS_LIST = []
text_train = []
text_test = []

file_list = sorted(os.listdir(FILE_DIR))

for file_name in file_list:
    m = re.match('\((.+)\) (\S+)_', file_name)
    if m:
        class_name, subset_name = m[1], m[2].lower()
        is_train = SIG_TRAIN in subset_name
        is_test = SIG_TEST in subset_name

        if is_train or is_test:
            if class_name not in CLASS_LIST:
                CLASS_LIST.append(class_name)
                text_train.append('')
                text_test.append('')

            cls = CLASS_LIST.index(class_name)
            with open(f'{FILE_DIR}/{file_name}', 'r') as f:
                text = f.read()

            subset = text_train if is_train else text_test
            subset[cls] += ' ' + text.replace('\n', ' ')

CLASS_COUNT = len(CLASS_LIST)
print(f'Загружено классов: {CLASS_COUNT}', CLASS_LIST)

<>:17: SyntaxWarning: invalid escape sequence '\('
<>:17: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_7644/1430949167.py:17: SyntaxWarning: invalid escape sequence '\('
  m = re.match('\((.+)\) (\S+)_', file_name)


Загружено классов: 6 ['Булгаков', 'Клиффорд_Саймак', 'Макс Фрай', 'О. Генри', 'Рэй Брэдберри', 'Стругацкие']


# **Токенизация и преобразование**

In [ ]:
# Параметры обработки
VOCAB_SIZE = 20000  # Объем словаря
WIN_SIZE   = 1000   # Длина окна (в словах)
WIN_HOP    = 100    # Шаг окна

# Обучение токенизатора
with timex():
    tokenizer = Tokenizer(num_words=VOCAB_SIZE,
                          filters='!"#$%&()*+,-–—./…:;<=>?@[\\]^_`{|}~«»\t\n\xa0\ufeff',
                          lower=True, split=' ', oov_token='неизвестное_слово')
    tokenizer.fit_on_texts(text_train)

    # Перевод текстов в последовательности индексов
    seq_train = tokenizer.texts_to_sequences(text_train)
    seq_test = tokenizer.texts_to_sequences(text_test)

print("Размер словаря:", len(tokenizer.word_index))

Время обработки: 7.18 с
Размер словаря: 133070


# **Нарезка данных**

In [ ]:
def split_sequence(sequence, win_size, hop):
    # Нарезка одной последовательности на отрезки
    return [sequence[i:i + win_size] for i in range(0, len(sequence) - win_size + 1, hop)]

def create_dataset(sequences, win_size, hop):
    x_data, y_data = [], []
    for cls, seq in enumerate(sequences):
        # Нарезаем последовательность класса на фрагменты
        segments = split_sequence(seq, win_size, hop)
        x_data.extend(segments)
        # Создаем метки (one-hot encoding)
        y_data.extend([utils.to_categorical(cls, CLASS_COUNT)] * len(segments))
    return np.array(x_data), np.array(y_data)

x_train, y_train = create_dataset(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = create_dataset(seq_test, WIN_SIZE, WIN_HOP)

print(f'Формат обучающей выборки: {x_train.shape}')

Формат обучающей выборки: (17640, 1000)


# **Создание и обучение модели**

In [ ]:
model = Sequential([
    Embedding(VOCAB_SIZE, 50, input_length=WIN_SIZE),
    SpatialDropout1D(0.2),
    Conv1D(64, 5, activation='relu'),
    MaxPooling1D(2),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(CLASS_COUNT, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Обучение
history = model.fit(x_train, y_train,
                    epochs=10,
                    batch_size=128,
                    validation_data=(x_test, y_test))

Epoch 1/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 265s 2s/step - accuracy: 0.4167 - loss: 1.4206 - val_accuracy: 0.4070 - val_loss: 1.4403
Epoch 2/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 261s 2s/step - accuracy: 0.6038 - loss: 0.9907 - val_accuracy: 0.3966 - val_loss: 1.5715
Epoch 3/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 259s 2s/step - accuracy: 0.7429 - loss: 0.6849 - val_accuracy: 0.4646 - val_loss: 1.3734
Epoch 4/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 255s 2s/step - accuracy: 0.8399 - loss: 0.4330 - val_accuracy: 0.5401 - val_loss: 1.4795
Epoch 5/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 263s 2s/step - accuracy: 0.8261 - loss: 0.4746 - val_accuracy: 0.5383 - val_loss: 1.5711
Epoch 6/10
 12/138 ━━━━━━━━━━━━━━━━━━━━ 3:29 2s/step - accuracy: 0.8856 - loss: 0.3114

KeyboardInterrupt: 

# **Визуализация**

In [ ]:
def plot_history(history):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Обучающая')
    plt.plot(history.history['val_accuracy'], label='Проверочная')
    plt.title('Точность')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Обучающая')
    plt.plot(history.history['val_loss'], label='Проверочная')
    plt.title('Потери')
    plt.legend()
    plt.show()

plot_history(history)

NameError: name 'history' is not defined